# 05 - Backtest & cost sensitivity

We run the full multi-factor strategy through the daily backtester with realistic costs,
then sweep three knobs to see what kills the alpha:
* commission + slippage (bps per side, on traded notional)
* short-borrow cost (bps annual, on average short notional)
* rebalance frequency (every N days)

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


In [ ]:
from mfa.data import forward_returns
from mfa.evaluation import quantile_returns, long_short_returns, information_coefficient
from mfa.evaluation.stats import perf_stats
from mfa.portfolio import combine_factors, factor_timing_weights, long_short_quantile_weights
from mfa.backtest import BacktestConfig, CostModel, run_backtest

# Build the standard composite + target weights
fwd1 = forward_returns(panels["close"], [1])[1]
fpnl = pd.DataFrame({n: long_short_returns(quantile_returns(f, fwd1, 5)) for n, f in factors.items()})
fic  = pd.DataFrame({n: information_coefficient(f, fwd1) for n, f in factors.items()})
fw   = factor_timing_weights(fpnl, fic, method="rolling_sharpe", lookback=63)
comp = combine_factors(factors, weights=fw, standardize=True)
print("composite shape:", comp.shape)


## 1. Baseline backtest

In [ ]:
target = long_short_quantile_weights(comp, top_q=0.2, bot_q=0.2, gross_leverage=2.0, max_weight=0.10)
daily_ret = panels["close"].pct_change().fillna(0.0)
bm = panels["benchmark_close"].iloc[:, 0].pct_change().fillna(0.0)

base_cfg = BacktestConfig(
    rebalance=5, initial_capital=1_000_000,
    cost_model=CostModel(commission_bps=1.0, slippage_bps=2.0, borrow_cost_bps_annual=50.0),
    benchmark_returns=bm,
)
bt = run_backtest(target, daily_ret, base_cfg)
perf = perf_stats(bt.net_returns)
pd.Series(perf, name="value").to_frame()


## 2. Trade-cost sweep (bps per side)

In [ ]:
def perf_for_costs(cps, sps, bps_borrow=50.0, rebal=5):
    cm = CostModel(commission_bps=cps, slippage_bps=sps, borrow_cost_bps_annual=bps_borrow)
    cfg = BacktestConfig(rebalance=rebal, initial_capital=1_000_000, cost_model=cm)
    bt = run_backtest(target, daily_ret, cfg)
    return perf_stats(bt.net_returns)

bps_grid = [0, 1, 2, 3, 5, 10, 20]
res = []
for tot in bps_grid:
    p = perf_for_costs(cps=tot/2, sps=tot/2, bps_borrow=50.0, rebal=5)
    res.append({"total_bps_per_side": tot, "sharpe": p["sharpe"], "ann_return": p["ann_return"], "max_dd": p["max_drawdown"]})
df = pd.DataFrame(res).set_index("total_bps_per_side")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["sharpe"].plot.bar(ax=ax[0], color="#2E86AB", edgecolor="black"); ax[0].set_title("Net Sharpe vs trade cost"); ax[0].axhline(0, color="black", lw=0.7)
df["ann_return"].plot.bar(ax=ax[1], color="#E63946", edgecolor="black"); ax[1].set_title("Annualised return vs trade cost"); ax[1].axhline(0, color="black", lw=0.7)
for a in ax: a.set_xlabel("commission+slippage per side (bps)")
plt.tight_layout(); plt.show()
df.round(3)


## 3. Rebalance-frequency sweep

In [ ]:
res = []
for rebal in [1, 2, 5, 10, 21]:
    p = perf_for_costs(cps=1.0, sps=2.0, bps_borrow=50.0, rebal=rebal)
    res.append({"rebalance_every_N_days": rebal, "sharpe": p["sharpe"], "ann_return": p["ann_return"], "max_dd": p["max_drawdown"]})
df = pd.DataFrame(res).set_index("rebalance_every_N_days")
fig, ax = plt.subplots(figsize=(8, 3.5))
df[["sharpe"]].plot.bar(ax=ax, color="#2E86AB", edgecolor="black"); ax.axhline(0, color="black", lw=0.7)
ax.set_title("Net Sharpe vs rebalance frequency"); plt.show()
df.round(3)


## 4. Borrow-cost sweep (basis points annual)

In [ ]:
res = []
for bc in [0, 25, 50, 100, 200]:
    p = perf_for_costs(cps=1.0, sps=2.0, bps_borrow=bc, rebal=5)
    res.append({"borrow_bps_annual": bc, "sharpe": p["sharpe"], "ann_return": p["ann_return"]})
df = pd.DataFrame(res).set_index("borrow_bps_annual")
fig, ax = plt.subplots(figsize=(8, 3.5))
df[["sharpe"]].plot.bar(ax=ax, color="#06A77D", edgecolor="black"); ax.axhline(0, color="black", lw=0.7)
ax.set_title("Net Sharpe vs short-borrow cost"); plt.show()
df.round(3)


## 5. Equity + drawdown, baseline strategy

In [ ]:
from mfa.viz import strategy_tearsheet
strategy_tearsheet(bt.net_returns, benchmark=bm, turnover=bt.turnover, perf=perf,
                   title="Multi-Factor strategy -- baseline backtest");


### Takeaways

* Net Sharpe degrades roughly linearly with trade cost; ~10 bps per side wipes most of the alpha.
* Rebalancing daily (rebal=1) maximises gross signal but turnover destroys net P&L.
* Borrow cost is a smaller knob than execution cost in this universe.